# Fraud Detection in Dollars, Not AUCMost notebooks on this competition optimize one number: ROC-AUC. But a fraudteam does not lose *AUC* when an attack slips through. It loses *money* - thetransaction amount. And it does not spend *AUC* reviewing a false alarm. Itspends an analyst's time.This notebook reframes the IEEE-CIS problem as a cost decision:- A missed fraud (false negative) costs the **transaction amount**.- A false alarm (false positive) costs a **fixed review fee**.Under that lens, the interesting output of a model is not its AUC but the**operating threshold** that minimizes total expected loss, and the **dollars**that threshold saves. A single, explainable LightGBM is enough to make thepoint; the value here is the framing, not a leaderboard rank.*Companion to a full portfolio project: cost-sensitive modeling, a FastAPIscoring service, drift monitoring, and a documented Kaggle investigation(pre-registered hypotheses, DeLong tests, honest nulls).*

## 1. The cost modelLet `c_fp` be the fixed cost of reviewing one false alarm (say 5 currencyunits of analyst time). For a decision threshold `t`, a transaction is flaggedwhen the model's fraud probability `p >= t`. The **Expected Monetary Loss**(EML) over a set of transactions is:```EML(t) = sum over missed frauds   ( transaction_amount )   # false negatives       + sum over false alarms     ( c_fp )                 # false positives```The baseline is the **approve-all** policy: flag nothing, so every fraud is aloss. EML(t) below that baseline is money saved. The cost-optimal threshold issimply `argmin_t EML(t)` - and because missing a large fraud is far moreexpensive than one review, that threshold sits far below the naive 0.5.

In [ ]:
import osfrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport lightgbm as lgbfrom sklearn.metrics import roc_auc_scoreRNG = 42C_FP = 5.0            # cost of one false-positive review, in currency unitsSPLIT_QUANTILE = 0.8  # temporal 80/20 split: train on the past, test on the future# The IEEE-CIS data is mounted read-only; locate it robustly across layouts.hits = sorted(Path("/kaggle/input").rglob("train_transaction.csv"))if not hits:    raise FileNotFoundError("Attach the IEEE-CIS competition data (Add Input -> Competitions).")DATA_DIR = hits[0].parentprint("Data dir:", DATA_DIR)

In [ ]:
train = pd.read_csv(DATA_DIR / "train_transaction.csv")try:    ident = pd.read_csv(DATA_DIR / "train_identity.csv")    df = train.merge(ident, on="TransactionID", how="left")except FileNotFoundError:    df = traindel trainfraud_rate = df["isFraud"].mean()total_fraud = df.loc[df["isFraud"] == 1, "TransactionAmt"].sum()print(f"Transactions : {len(df):,}")print(f"Fraud rate   : {fraud_rate:.2%}  (about 1 in {int(round(1 / fraud_rate))})")print(f"Fraud dollars: {total_fraud:,.0f}  <- what a fraud team actually loses if it does nothing")

## 2. A model - kept deliberately simpleThe point of this notebook is the decision layer, so the model is onegradient-boosted tree ensemble (LightGBM) on a compact, honest feature set:- the numeric base (transaction + identity columns), with **missing values left  as NaN** - LightGBM routes them natively, which keeps "missing" distinct from  a real zero;- **categorical encodings** (label + frequency) fit on the training rows only;- **row-local** time, amount, and D-column features.Every encoder is fit on the past (training) partition and applied unchanged tothe future (holdout) partition - the same discipline a production system needs,where you cannot peek at future transactions.

In [ ]:
MISSING = "__missing__"FREQ_NUMERIC = ["card1", "card2", "card3", "card5", "addr1", "addr2"]D_NORM = [f"D{i}" for i in range(1, 16) if i != 9]  # D9 is an hour fraction, not a day counterEXCLUDE = {"isFraud", "TransactionID", "TransactionDT"}def as_str(s):    return s.astype("object").where(s.notna(), MISSING).astype(str)def freq_encode(train_vals, vals):    table = as_str(train_vals).value_counts(normalize=True)    return as_str(vals).map(table).fillna(0.0).astype("float32")def label_encode(train_vals, vals):    codes = {v: i for i, v in enumerate(sorted(as_str(train_vals).unique()))}    return as_str(vals).map(codes).fillna(-1).astype("int32")def split_email(vals, prefix):    parts = as_str(vals).str.split(".")    return pd.DataFrame({f"{prefix}_provider": parts.str[0], f"{prefix}_suffix": parts.str[-1]},                        index=vals.index)def time_features(dt):    return pd.DataFrame({"tx_hour": ((dt // 3600) % 24).astype("int32"),                         "tx_dow": ((dt // 86400) % 7).astype("int32")}, index=dt.index)def amount_features(amt):    cents = (amt - np.floor(amt)).round(2)    return pd.DataFrame({"amt_log1p": np.log1p(amt).astype("float32"),                         "amt_cents": cents.astype("float32")}, index=amt.index)def normalize_d(frame, dt, cols):    days = dt / 86400.0    out = pd.DataFrame(index=frame.index)    for c in cols:        if c in frame.columns:            out[f"{c}_norm"] = (frame[c] - days).astype("float32")    return outprint("Feature functions ready.")

In [ ]:
# Temporal split: the future must not leak into the past.cutoff = df["TransactionDT"].quantile(SPLIT_QUANTILE)train_mask = (df["TransactionDT"] < cutoff).to_numpy()for col, prefix in [("P_emaildomain", "P_email"), ("R_emaildomain", "R_email")]:    if col in df.columns:        df = pd.concat([df, split_email(df[col], prefix)], axis=1)numeric = [c for c in df.columns if df[c].dtype != "O" and c not in EXCLUDE]label_cols = [c for c in df.columns if df[c].dtype == "O"]freq_cols = label_cols + [c for c in FREQ_NUMERIC if c in df.columns]row_local = pd.concat([time_features(df["TransactionDT"]),                       amount_features(df["TransactionAmt"]),                       normalize_d(df, df["TransactionDT"], D_NORM)], axis=1)X_num = pd.concat([df[numeric].astype("float32"), row_local], axis=1).reset_index(drop=True)# Encoders fit on the training partition only.enc = pd.DataFrame(index=df.index)for c in label_cols:    enc[f"{c}_le"] = label_encode(df.loc[train_mask, c], df[c])for c in freq_cols:    enc[f"{c}_freq"] = freq_encode(df.loc[train_mask, c], df[c])X = pd.concat([X_num, enc.reset_index(drop=True)], axis=1)y = df["isFraud"].astype(int).to_numpy()amount = df["TransactionAmt"].fillna(0.0).to_numpy()print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")

In [ ]:
params = dict(objective="binary", learning_rate=0.05, num_leaves=192,              min_data_in_leaf=100, feature_fraction=0.8, bagging_fraction=0.8,              bagging_freq=1, seed=RNG, n_jobs=-1, verbosity=-1)model = lgb.LGBMClassifier(n_estimators=800, **params)model.fit(X[train_mask], y[train_mask])holdout_proba = model.predict_proba(X[~train_mask])[:, 1]holdout_auc = roc_auc_score(y[~train_mask], holdout_proba)print(f"Holdout ROC-AUC: {holdout_auc:.4f}  (temporal, most recent 20%)")

## 3. From probabilities to a decisionAUC summarizes ranking across *all* thresholds. But a fraud system has to pick*one*. The chart below sweeps the threshold and plots EML at each point. Theminimum is the cost-optimal operating point - and it sits near the extremeleft, because the asymmetry (a missed fraud costs its full amount, a falsealarm costs `c_fp = 5`) rewards aggressive flagging.A subtle but expensive detail: a coarse grid starting at 0.01 **misses the trueoptimum**. A well-separated model concentrates its scores near zero, so thegrid has to be fine below 0.01 to find the real minimum.

In [ ]:
def expected_loss(y_true, proba, amt, c_fp, t):    yhat = (proba >= t).astype(int)    fn = (y_true == 1) & (yhat == 0)    fp = (y_true == 0) & (yhat == 1)    return float((fn * amt).sum() + fp.sum() * c_fp)def approve_all(y_true, amt):    return float(((y_true == 1) * amt).sum())y_h = y[~train_mask]amt_h = amount[~train_mask]# Fine grid below 0.01 (where the optimum hides), then the classic sweep.grid = np.concatenate([np.arange(0.001, 0.01, 0.0005), np.linspace(0.01, 0.99, 99)])losses = np.array([expected_loss(y_h, holdout_proba, amt_h, C_FP, t) for t in grid])baseline = approve_all(y_h, amt_h)best_i = int(losses.argmin())best_t, best_loss = float(grid[best_i]), float(losses[best_i])plt.figure(figsize=(11, 5))plt.plot(grid, losses, lw=2, label="Expected Monetary Loss")plt.axhline(baseline, ls="--", color="crimson", label=f"Approve-all baseline  ${baseline:,.0f}")plt.axvline(best_t, ls=":", color="orange", label=f"Optimal threshold  t={best_t:.3f}")plt.scatter([best_t], [best_loss], color="green", zorder=5, s=90,            label=f"Minimum loss  ${best_loss:,.0f}")plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))plt.xlabel("Decision threshold")plt.ylabel("Expected Monetary Loss")plt.title("The decision that matters: threshold vs. dollars")plt.legend()plt.grid(alpha=0.3)plt.tight_layout()plt.show()print(f"Cost-optimal threshold: {best_t:.4f}")print(f"Loss at optimum       : ${best_loss:,.0f}")print(f"Reduction vs approve-all: ${baseline - best_loss:,.0f}  ({(baseline - best_loss) / baseline:.1%})")

## 4. The monetary waterfallThe same result, told in the language a business understands: start from theapprove-all loss, subtract what the model prevents, add back the operationalcost of the false alarms it generates, and you land on the residual loss.

In [ ]:
model_loss = expected_loss(y_h, holdout_proba, amt_h, C_FP, best_t)fn_only = expected_loss(y_h, holdout_proba, amt_h, 0.0, best_t)  # missed-fraud dollars onlyfp_cost = model_loss - fn_only                                    # review cost onlyprevented = baseline - model_losslabels = ["Approve-all\nbaseline", "Prevented\nby model", "False-alarm\nreview cost", "Residual\nloss"]values = [baseline, prevented, fp_cost, model_loss]colors = ["crimson", "seagreen", "orange", "steelblue"]plt.figure(figsize=(10, 5))bars = plt.bar(labels, values, color=colors)for b, v in zip(bars, values):    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + baseline * 0.01,             f"${v:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))plt.title(f"Where the money goes at t={best_t:.3f}")plt.ylabel("Currency units")plt.grid(axis="y", alpha=0.3)plt.tight_layout()plt.show()flagged = (holdout_proba >= best_t)recall = y_h[flagged].sum() / y_h.sum()precision = y_h[flagged].sum() / flagged.sum() if flagged.sum() else 0.0print(f"Recall (fraud caught): {recall:.1%}")print(f"Precision (alert is real fraud): {precision:.1%}  vs {fraud_rate:.1%} for a random queue")

## 5. How the score was built - one block at a timeThis model did not appear at once. In the full project each feature block wasadded on its own and scored on the frozen leaderboard, so every gain isattributable (public/private LB, late submissions):| Step | Block | Private LB ||---|---|---|| EXP-000 | production baseline (numeric-only, sklearn GB) | 0.8749 || EXP-001 | LightGBM + native NaN | 0.8877 || EXP-002 | categorical encodings | 0.8968 || EXP-003 | time / amount / D-normalization | 0.8998 || EXP-004 | minimal UID key | 0.9032 || EXP-006 | full aggregation engine | 0.9078 || EXP-007 | temporal-stability selection | 0.9077 |Total: **0.8749 -> 0.9078** with a single explainable model. No block was the"magic" one; the gain is broad and incremental.Two honest findings matter more than the digit:1. **Internal validation overstates gains under drift.** Every block's gain on   the recent holdout shrank on the private (later) test period. Selecting on a   naive most-recent-slice holdout tracked the private leaderboard better than a   more elaborate month-wise cross-validation did.2. **The ceiling is entity memorization.** Holdout AUC was ~0.99 on cards seen   in training vs ~0.90 on new cards. The private test set is mostly new cards,   so the private score approaches the new-card ceiling. The gap to the top   single models (~0.93) is not more features - it is entity linkage (connecting   test transactions to known cards), which the winners engineered at length.

## 6. TakeawayA model's AUC is a ranking statistic. A fraud program's success is a dollarfigure. Once you commit to the cost model, three things follow that anAUC-first notebook never surfaces:- the **operating threshold** is a business decision, not a default of 0.5;- that threshold can hide **below a coarse grid** - here it sat near 0.003, and  a grid starting at 0.01 quietly left money on the table;- the honest headline is **dollars prevented and the residual loss**, not a  leaderboard position.A single, explainable LightGBM reaches a private LB around the median of the6,381 teams - deliberately without the ensembling and entity-linkage the topsolutions used. The competitive rank was never the goal; a decision system youcan defend in dollars was.*If this framing was useful, an upvote is appreciated. Questions and critiqueswelcome in the comments.*